# EX3 Logistic Regression: Breast Cancer

In [ ]:
#%pip install scikit-learn

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as opt
import numpy as np
from matplotlib import pyplot as plt

## data preprocessing

In [ ]:
dataset = load_breast_cancer()
X,y = dataset.data,dataset.target

In [ ]:
X[0]

In [ ]:
y[0]

train test split

In [ ]:
print(f'X shape {X.shape} y shape {y.shape}')
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.33)

normalize data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

convert to torch tensors

In [ ]:
y_train = y_train.reshape(-1,1)
y_test = y_test.reshape(-1,1)
train_inputs = torch.from_numpy(X_train.astype(np.float32))
test_inputs = torch.from_numpy(X_test.astype(np.float32))
train_targets = torch.from_numpy(y_train.astype(np.float32))
test_targets = torch.from_numpy(y_test.astype(np.float32))

## building the model

In [ ]:
D = X_test.shape[1]

model = nn.Sequential(
    nn.Linear(D,1),
    nn.Sigmoid()
)
optimizer = opt.Adam(model.parameters(),lr=0.01)
criterion = nn.BCELoss()

## training the model

In [ ]:
EPOCHS = 700
train_losses = []
test_losses= []
train_accuracies = []
test_accuracies = []

In [ ]:
for epoch in range(EPOCHS):
    model.zero_grad()

    predictions = model(train_inputs)
    loss = criterion(predictions,train_targets)
    train_losses.append(loss.item())
    loss.backward()
    optimizer.step()

    #train acc
    train_preds = np.round(predictions.detach().numpy())
    train_acc = np.mean(train_preds == train_targets.detach().numpy())
    train_accuracies.append(train_acc.item())

    #test loss
    outs_test = model(test_inputs)
    test_loss = criterion(outs_test,test_targets)
    test_losses.append(test_loss.item())

    #test acc
    test_preds = np.round(outs_test.detach().numpy())
    test_acc = np.mean(test_preds == test_targets.detach().numpy())
    test_accuracies.append(test_acc.item())
    #test loss every 50 epochs
    if (epoch + 1) % 50 == 0:
       
        print(f'{epoch +1}/{EPOCHS} train loss: {loss.item():.4f} test loss: {test_loss.item():.4f} train acc: {train_acc.item():.4f} val acc: {test_acc.item():.4f}')
 

## plot the loss and accuracy

In [ ]:

epochs = [i for i in range(len(train_losses))]

In [ ]:
plt.plot(epochs,train_losses,label='train loss',color='red')
plt.plot(epochs,test_losses,label='test loss',color='orange')
plt.legend()
plt.xlabel = 'epochs'
plt.show()

In [ ]:
plt.plot(epochs,train_accuracies,label='train acc',color='blue')
plt.plot(epochs,test_accuracies,label='test acc',color='green')
plt.legend()
plt.xlabel = 'epochs'
plt.show()

## predict

In [ ]:
with torch.no_grad():
    preds_train = model(train_inputs)
    preds_train = np.round(preds_train.numpy()) #since it's probability it's not exactly 1 and 0 yet os we round it
    train_acc = np.mean(train_targets.numpy() == preds_train)
# train_targets.numpy() == preds_train
# Creates a vector of true/false but numpy treat True as 1 and false as 0
# So we can just count the number of 1s or JUST ADD THEM to get
# the number of correct predictions! But mean also divide it with th TOTAL NUMBER OF PREDICTIONS thus we got accuracy

    pred_test = model(test_inputs)
    pred_test = np.round(pred_test.numpy())
    test_acc = np.mean(test_targets.numpy() == pred_test)
    print(f'train acc {train_acc} test acc {test_acc}')
    

# Saving and Loading a Model

In [ ]:
model.state_dict()

In [ ]:
torch.save(model.state_dict(),'../MODELS/cancer_classification.pt')